In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ctf_dataset.load import create_wrapped_dataset
from os.path import join
from coupling_metrics import phase_synchrony, isps, cofluctuation, iscf, window_isc, window_isfc
import scipy
from itertools import combinations
import pycwt
import pandas as pd
import seaborn as sns
import sklearn

from sklearn import mixture
import statistics

from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score

from itertools import combinations
from sklearn.model_selection import PredefinedSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from scipy.stats import zscore

from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

from scipy.stats import ttest_1samp

base_dir = '/jukebox/hasson/vgraf/social-ctf'
data_dir = join(base_dir, 'data')

wrap_f = create_wrapped_dataset(data_dir, output_dataset_name="virtual.hdf5")
lstmPCs = np.load('results/lstms_tanh-z_pca-k100.npy')

datadir = "/jukebox/hasson/snastase/social-ctf/results"
lstmsNew = np.load(f'{datadir}/lstms-pca_matchup-0_map-26_repeat-8.npy') 

In [2]:
from scipy.stats import zscore

n_repeats = 32
n_maps = 32
n_samples = 4501
n_players = 4
n_pairs = n_players * (n_players - 1) / 2
pairs = combinations(np.arange(n_players), 2)
n_pcs = 142

# data shape = (2 repeats, 4 players, 4501 samples, 142 PCs)
lstms = np.full((n_maps, n_repeats, n_samples, n_players, n_pcs), np.nan)
map_ids = []
for map_id in np.arange(n_maps):
    for repeat_id in np.arange(n_repeats):
        data = np.load(f'{datadir}/lstms-pca_matchup-0_map-{map_id}_repeat-{repeat_id}.npy')
        lstms[map_id, repeat_id] = zscore(np.moveaxis(data[..., :n_pcs], 0, 1), axis=0)
        map_ids.extend([map_id]*n_samples)
    print(f'Finished loading {map_id}')
    
lstms_stack = np.dstack(np.split(lstms, n_repeats, axis=1)).squeeze() #32 x 144032 x 2
lstms_stack = np.hstack(np.split(lstms_stack, n_maps, axis=0)).squeeze()
print(f"LSTMs stack shape: {lstms_stack.shape}")

MemoryError: Unable to allocate 19.5 GiB for an array with shape (32, 32, 4501, 4, 142) and data type float64

In [3]:
results = np.load('regression_results.npy')

In [4]:
results[141]

array([[ 0.05484523,  0.13323616,  0.06903976, ...,  0.03356966,
         0.08837673,  0.06913813],
       [-0.04155279, -0.04875029,  0.02500249, ..., -0.00250315,
         0.0011494 ,  0.03771705],
       [-0.01373049, -0.03090669, -0.00588556, ...,  0.02879681,
         0.02910099, -0.00147247],
       [-0.00607342, -0.0252036 ,  0.03664364, ..., -0.0043242 ,
         0.00531012,  0.06921938],
       [-0.06142289, -0.03985055, -0.00029069, ...,  0.00136726,
        -0.03633371,  0.06467134],
       [ 0.11045286,  0.08290671,  0.12158667, ...,  0.06162716,
         0.01700426,  0.03507014]])